In [3]:
%pip install traffic

import pandas as pd
import os
from pyproj import Geod 
import numpy as np
from traffic.core import Flight


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.13.3 which is incompatible.


INFO: pip is looking at multiple versions of pydantic to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/28.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/28.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/28.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/28.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/28.3 MB ? eta -:--:--
    --------------------------------------- 0.5/28.3 MB 479.2 kB/s eta 0:00:59
    --------------------------------------- 0.5/28.3 MB 479.2 kB/s eta 0:00:59
    --------------------------------------- 0.5/28.3 MB 479.2 kB/s eta 0:00:59
   - -------------------------------------- 0.8/28.3 MB 447.3 kB/s eta 0:01:02
   - -------------------------------------- 0.8/28.3 MB 447.3 kB/s eta 0:01:02
   - -------------------------------------- 0.8/28.3 MB 447.3 kB/s eta 0:01:02
   - --------------------------

# Helper Functions

In [4]:
def interpolate_sparse_adsc(df, dt=15.0, alt_before=None, alt_after=None):
    from pyproj import Geod
    geod = Geod(ellps="WGS84")
    
    df = df.sort_values("timestamp").reset_index(drop=True)
    
    # Build altitude array
    alt_col = "altitude_m" if "altitude_m" in df.columns else "altitude"
    known_alts = pd.to_numeric(df[alt_col], errors="coerce").values.copy()
    if alt_before is not None and np.isnan(known_alts[0]):
        known_alts[0] = alt_before
    if alt_after is not None and np.isnan(known_alts[-1]):
        known_alts[-1] = alt_after
    s = pd.Series(known_alts).interpolate(method="linear").bfill().ffill()
    known_alts = s.values
    
    all_points = []
    for i in range(len(df) - 1):
        lat1, lon1 = df.iloc[i]["latitude"], df.iloc[i]["longitude"]
        lat2, lon2 = df.iloc[i+1]["latitude"], df.iloc[i+1]["longitude"]
        alt1, alt2 = known_alts[i], known_alts[i+1]
        t1, t2 = df.iloc[i]["timestamp"], df.iloc[i+1]["timestamp"]
        
        gap_seconds = (t2 - t1).total_seconds()
        
        # Number of intermediate points that fit before t2
        n_intermediate = max(0, int(gap_seconds / dt) - 1)
        
        # Add start point
        all_points.append({
            "timestamp": t1,
            "latitude": lat1,
            "longitude": lon1,
            "altitude": alt1,
            "interpolated": False
        })
        
        if n_intermediate > 0:
            points = geod.npts(lon1, lat1, lon2, lat2, n_intermediate + 1)
            # Take only n_intermediate points (skip the last one which is near t2)
            for j in range(n_intermediate):
                frac = (j + 1) / (n_intermediate + 2)
                alt_i = alt1 + frac * (alt2 - alt1)
                t_i = t1 + pd.Timedelta(seconds=dt * (j + 1))
                all_points.append({
                    "timestamp": t_i,
                    "latitude": points[j][1],
                    "longitude": points[j][0],
                    "altitude": alt_i,
                    "interpolated": True
                })
    
    # Add last point
    all_points.append({
        "timestamp": df.iloc[-1]["timestamp"],
        "latitude": df.iloc[-1]["latitude"],
        "longitude": df.iloc[-1]["longitude"],
        "altitude": known_alts[-1],
        "interpolated": False
    })
    
    return pd.DataFrame(all_points)

In [5]:
import numpy as np
import pandas as pd

def clean_flight(df: pd.DataFrame, min_points: int = 20) -> pd.DataFrame:
    """
    Clean a single flight dataframe.
    Returns cleaned dataframe or empty dataframe if flight is unusable.
    """
    df = df.copy()

    # 1. Drop exact duplicate rows
    df = df.drop_duplicates()

    # 2. Sort by timestamp
    df = df.sort_values("timestamp").reset_index(drop=True)

    # 3. Drop rows missing critical columns
    df = df.dropna(subset=["timestamp", "latitude", "longitude"])

    # 4. Remove on-ground points
    df = df[df["onground"] != True].reset_index(drop=True)

    # 5. Remove physically unrealistic values
    df = df[
        (df["latitude"].between(-90, 90))
        & (df["longitude"].between(-180, 180))
        & (df["geoaltitude_m"].fillna(0) >= 0)
        & (df["baroaltitude_m"].fillna(0) >= 0)
        & (df["velocity_mps"].fillna(0) >= 0)
        & (df["velocity_mps"].fillna(0) <= 340)
        & (df["heading_deg"].fillna(0) >= 0)
        & (df["heading_deg"].fillna(360) < 360)
    ].reset_index(drop=True)

    # 6. Deduplicate timestamps (keep first)
    df = df.drop_duplicates(subset=["timestamp"], keep="first").reset_index(drop=True)

    # 7. Strip callsign whitespace
    if "callsign" in df.columns:
        df["callsign"] = df["callsign"].astype(str).str.strip()

    # 8. Reconcile altitude: use baroaltitude, fill gaps from geoaltitude
    df["altitude"] = df["baroaltitude_m"].fillna(df["geoaltitude_m"])

    # 9. Remove speed jumps (implied speed between consecutive points)
    if len(df) > 1:
        lat1 = np.radians(df["latitude"].values[:-1])
        lat2 = np.radians(df["latitude"].values[1:])
        lon1 = np.radians(df["longitude"].values[:-1])
        lon2 = np.radians(df["longitude"].values[1:])
        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
        haversine_dist = 2 * 6371000 * np.arcsin(np.sqrt(a))

        dt = np.diff(df["timestamp"].values).astype(float)
        # avoid division by zero
        dt = np.where(dt == 0, 1e-6, dt)

        implied_speed = haversine_dist / dt
        # mark points where implied speed to next point > 400 m/s
        bad_mask = np.concatenate([[False], implied_speed > 400])
        df = df[~bad_mask].reset_index(drop=True)

    # 10. Remove altitude jumps
    if len(df) > 1:
        alt = df["altitude"].values
        dt = np.diff(df["timestamp"].values).astype(float)
        dt = np.where(dt == 0, 1e-6, dt)
        alt_diff = np.abs(np.diff(alt))
        # flag points with > 5000m altitude change AND short dt (< 30s)
        bad_alt = np.concatenate([[False], (alt_diff > 5000) & (dt < 30)])
        df = df[~bad_alt].reset_index(drop=True)

    # 11. Interpolate small gaps in non-critical columns
    for col in ["velocity_mps", "heading_deg", "altitude"]:
        df[col] = df[col].interpolate(method="linear", limit=5)

    # 12. Remove too-short flights
    if len(df) < min_points:
        return pd.DataFrame(columns=df.columns)

    return df.reset_index(drop=True)

In [6]:
import pandas as pd
import numpy as np
from pyproj import Geod

geod = Geod(ellps="WGS84")

def add_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """Add time-based and kinematic features from resampled positions."""
    df = df.copy().sort_values("timestamp").reset_index(drop=True)

    # Time delta
    df["delta_t"] = df["timestamp"].diff().dt.total_seconds()
    df.loc[df["delta_t"] == 0, "delta_t"] = np.nan

    # Recompute velocity and heading from lat/lon
    lat1 = df["latitude"].values[:-1]
    lon1 = df["longitude"].values[:-1]
    lat2 = df["latitude"].values[1:]
    lon2 = df["longitude"].values[1:]

    az12, _, dist = geod.inv(lon1, lat1, lon2, lat2)
    az12 = az12 % 360

    dt = df["delta_t"].values[1:]

    df["heading_deg"] = np.concatenate([[np.nan], az12])
    df["velocity_mps"] = np.concatenate([[np.nan], dist / dt])

    # Velocity components
    heading_rad = np.deg2rad(df["heading_deg"])
    df["vx"] = df["velocity_mps"] * np.sin(heading_rad)
    df["vy"] = df["velocity_mps"] * np.cos(heading_rad)

    # Acceleration and turn rate
    df["acceleration"] = df["velocity_mps"].diff() / df["delta_t"]
    df["turn_rate"] = df["heading_deg"].diff() / df["delta_t"]

    return df

# Entire process

In [7]:
output_folder = "cleaned_data_final"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
base_folder = "raw_data"

dirs = [d for d in os.listdir(base_folder) if os.path.isdir(os.path.join(base_folder, d))]
cur_directory = os.path.join(base_folder, dirs[0],"flights")

for d in dirs:
    cur_directory = os.path.join(base_folder, d,"flights")
    if not os.path.exists(cur_directory):
        os.makedirs(cur_directory)

    flights = [f for f in os.listdir(cur_directory) if os.path.isdir(os.path.join(base_folder, d))]
    for f in flights:
        cur_flight = os.path.join(cur_directory, f)
        ads_b_before = os.path.join(cur_flight, "adsb_before.parquet")
        ads_b_after = os.path.join(cur_flight, "adsb_after.parquet")
        ads_c = os.path.join(cur_flight, "adsc.parquet")

        dfs = {
            "adsb_before": pd.read_parquet(ads_b_before),
            "adsb_after": pd.read_parquet(ads_b_after),
            "adsc": pd.read_parquet(ads_c)
        }
        # create directory for new flights if it doesn't exist
        
        cur_output_folder = os.path.join(output_folder,d, f)
        if not os.path.exists(cur_output_folder):
            os.makedirs(cur_output_folder)
        for key in ["adsb_before", "adsb_after", "adsc"]:
            df = dfs[key]
            
            if key == "adsb_before":
                alt_before = df["geoaltitude_m"].iloc[-1]
            if key == "adsb_after":
                alt_after = df["geoaltitude_m"].iloc[0]
            if key == "adsc":
                
                df = interpolate_sparse_adsc(df, alt_before=alt_before, alt_after=alt_after)
                df = df.set_index("timestamp").resample("15s").first().interpolate().reset_index()
                cleaned_df = df
            
            else:
                cleaned_df = clean_flight(df)
                if cleaned_df.empty:
                    continue
                cleaned_df = Flight(cleaned_df).resample("15S").data
            
            cleaned_df = add_feature_engineering(cleaned_df)
            output_path = os.path.join(cur_output_folder, f"{key}.parquet")
            #print(output_path)
            cleaned_df.to_parquet(output_path)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'raw_data'